In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
matches_df = pd.read_csv("../data/processed/matches.csv")
matches_df.head()

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,total_goals,total_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750,8,4.770630
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787,2,3.088067
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460,4,2.465610
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200,5,5.311210
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060,5,2.549952


In [3]:
teams_df = pd.read_csv("../data/processed/teams.csv")
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [4]:
home_df = (teams_df[teams_df['h_a']=='h'].groupby(['team_name','league','year'])
           [['scored','missed','xG','xGA']].sum().reset_index())

home_df.columns = ['team_name','league','year','scored_h','conceded_h','xG_h','xGA_h']

away_df = (teams_df[teams_df['h_a']=='a'].groupby(['team_name','league','year'])
           [['scored','missed','xG','xGA']].sum().reset_index())

away_df.columns = ['team_name','league','year','scored_a','conceded_a','xG_a','xGA_a']


In [5]:
seasonal_df = home_df.merge(away_df, on=['team_name','league','year'])

seasonal_df['scored'] = seasonal_df['scored_h'] + seasonal_df['scored_a']
seasonal_df['conceded'] = seasonal_df['conceded_h'] + seasonal_df['conceded_a']
seasonal_df['xG'] = seasonal_df['xG_h'] + seasonal_df['xG_a']
seasonal_df['xGA'] = seasonal_df['xGA_h'] + seasonal_df['xGA_a']

seasonal_df['xGD_Attack'] = seasonal_df['scored'] - seasonal_df['xG']
seasonal_df['xGD_Defence'] = seasonal_df['conceded'] - seasonal_df['xGA']

seasonal_df['xGD_Attack_h'] = seasonal_df['scored_h'] - seasonal_df['xG_h']
seasonal_df['xGD_Attack_a'] = seasonal_df['scored_a'] - seasonal_df['xG_a']
seasonal_df['xGD_Defence_h'] = seasonal_df['conceded_h'] - seasonal_df['xGA_h']
seasonal_df['xGD_Defence_a'] = seasonal_df['conceded_a'] - seasonal_df['xGA_a']

In [6]:
seasonal_df.head()

,team_name,league,year,scored_h,conceded_h,xG_h,xGA_h,scored_a,conceded_a,xG_a,...,scored,conceded,xG,xGA,xGD_Attack,xGD_Defence,xGD_Attack_h,xGD_Attack_a,xGD_Defence_h,xGD_Defence_a
0,AC Milan,serie_a,2020,31,24,37.928325,24.530668,43,17,37.119001,...,74,41,75.047326,48.838073,-1.047326,-7.838073,-6.928325,5.880999,-0.530668,-7.307405
1,AC Milan,serie_a,2021,28,12,32.436844,14.717525,41,19,34.901264,...,69,31,67.338108,34.631340,1.661892,-3.631340,-4.436844,6.098736,-2.717525,-0.913815
2,AC Milan,serie_a,2022,41,20,40.158294,17.496600,23,23,29.742495,...,64,43,69.900789,40.109474,-5.900789,2.890526,0.841706,-6.742495,2.503400,0.387126
3,AC Milan,serie_a,2023,38,17,37.496432,21.897890,38,32,34.359323,...,76,49,71.855755,47.731906,4.144245,1.268094,0.503568,3.640677,-4.897890,6.165984
4,AC Milan,serie_a,2024,30,16,31.668365,20.520671,31,27,37.841776,...,61,43,69.510141,47.271499,-8.510141,-4.271499,-1.668365,-6.841776,-4.520671,0.249172


In [7]:
seasonal_df[seasonal_df['team_name'] == 'Bayern Munich'][['scored','conceded','xG','xGA']]

,scored,conceded,xG,xGA
60,99,44,75.932410,38.873343
61,97,37,99.905525,38.609638
62,92,38,85.118184,37.006959
63,94,45,94.400968,35.224795
64,99,32,95.017114,27.987424


In [8]:
seasonal_df[(seasonal_df['team_name'] == 'Bayern Munich')]

,team_name,league,year,scored_h,conceded_h,xG_h,xGA_h,scored_a,conceded_a,xG_a,...,scored,conceded,xG,xGA,xGD_Attack,xGD_Defence,xGD_Attack_h,xGD_Attack_a,xGD_Defence_h,xGD_Defence_a
60,Bayern Munich,bundesliga,2020,64,21,46.027431,18.501977,35,23,29.904979,...,99,44,75.932410,38.873343,23.067590,5.126657,17.972569,5.095021,2.498023,2.628634
61,Bayern Munich,bundesliga,2021,48,15,49.282000,19.077494,49,22,50.623525,...,97,37,99.905525,38.609638,-2.905525,-1.609638,-1.282000,-1.623525,-4.077494,2.467856
62,Bayern Munich,bundesliga,2022,53,17,49.365890,16.183654,39,21,35.752294,...,92,38,85.118184,37.006959,6.881816,0.993041,3.634110,3.247706,0.816346,0.176695
63,Bayern Munich,bundesliga,2023,53,12,52.129240,13.131342,41,33,42.271728,...,94,45,94.400968,35.224795,-0.400968,9.775205,0.870760,-1.271728,-1.131342,10.906547
64,Bayern Munich,bundesliga,2024,53,16,55.768840,13.508277,46,16,39.248274,...,99,32,95.017114,27.987424,3.982886,4.012576,-2.768840,6.751726,2.491723,1.520853


In [9]:
matches_df

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,total_goals,total_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750,8,4.770630
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787,2,3.088067
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460,4,2.465610
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200,5,5.311210
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060,5,2.549952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8977,2025-05-25 18:45:00,serie_a,2024,27733,107,Atalanta,ATA,230,Parma Calcio 1913,PAR,2,3,1.902650,2.825350,5,4.728000
8978,2025-05-25 18:45:00,serie_a,2024,27736,108,Empoli,EMP,94,Verona,VER,1,2,1.233510,0.833807,3,2.067317
8979,2025-05-25 18:45:00,serie_a,2024,27737,96,Lazio,LAZ,243,Lecce,LEC,0,1,1.735210,0.708667,1,2.443877
8980,2025-05-25 18:45:00,serie_a,2024,27740,113,Torino,TOR,95,Roma,ROM,0,2,0.381059,2.644360,2,3.025419


In [10]:
teams_df['date'] = pd.to_datetime(teams_df['date'])
matches_df['datetime'] = pd.to_datetime(matches_df['datetime'])

clinical_df = teams_df.merge(
    matches_df[['datetime','league','year','match_id','home_team','away_team']],
    left_on = ['date','league','year'],
    right_on = ['datetime','league','year'],
    how = 'left'
)

In [11]:
clinical_df

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,pts,npxGD,team_id,team_name,league,year,datetime,match_id,home_team,away_team
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,3,3.713360,117,Bayern Munich,bundesliga,2020,2020-09-18 18:30:00,14173,Bayern Munich,Schalke 04
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,0,-0.592510,117,Bayern Munich,bundesliga,2020,2020-09-27 13:30:00,14189,Hoffenheim,Bayern Munich
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,3,1.990240,117,Bayern Munich,bundesliga,2020,2020-10-04 16:00:00,15166,Bayern Munich,Hertha Berlin
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,3,2.082120,117,Bayern Munich,bundesliga,2020,2020-10-17 16:30:00,15172,Arminia Bielefeld,Bayern Munich
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,3,3.379001,117,Bayern Munich,bundesliga,2020,2020-10-24 13:30:00,15177,Union Berlin,Freiburg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39879,a,1.12443,1.554050,1.12443,1.554050,"{'att': 85, 'def': 13}","{'att': 346, 'def': 24}",7,1,1,...,1,-0.429620,286,Como,serie_a,2024,2025-05-18 18:45:00,27729,Parma Calcio 1913,Napoli
39880,a,1.12443,1.554050,1.12443,1.554050,"{'att': 85, 'def': 13}","{'att': 346, 'def': 24}",7,1,1,...,1,-0.429620,286,Como,serie_a,2024,2025-05-18 18:45:00,27730,Roma,AC Milan
39881,a,1.12443,1.554050,1.12443,1.554050,"{'att': 85, 'def': 13}","{'att': 346, 'def': 24}",7,1,1,...,1,-0.429620,286,Como,serie_a,2024,2025-05-18 18:45:00,27731,Juventus,Udinese
39882,h,1.49032,2.277890,1.49032,2.277890,"{'att': 483, 'def': 15}","{'att': 249, 'def': 19}",8,8,0,...,0,-0.787570,286,Como,serie_a,2024,2025-05-23 18:45:00,27735,Como,Inter


In [ ]:
# Finishing Attributes (per team)
clinical_df['match_xGD']     = clinical_df['scored'] - clinical_df['xG']
clinical_df['match_npxGD']   = clinical_df['scored'] - clinical_df['npxG']
clinical_df['xGA_diff']      = clinical_df['missed'] - clinical_df['xGA']
clinical_df['dominance']     = clinical_df['xG']     - clinical_df['xGA']

clinical_df['clinical']       = np.where((clinical_df['match_xGD'] > 0)     & (clinical_df['xG'] > 0.5), 'clinical',       'not_clinical')
clinical_df['ultra_clinical'] = np.where((clinical_df['match_xGD'] >= 1.5)  & (clinical_df['xG'] > 0.5), 'ultra_clinical',  'not_ultra_clinical')
clinical_df['wasteful']       = np.where((clinical_df['match_xGD'] < 0)     & (clinical_df['xG'] > 0.5), 'wasteful',        'not_wasteful')
clinical_df['ultra_wasteful'] = np.where((clinical_df['match_xGD'] <= -1.5) & (clinical_df['xG'] > 0.5), 'ultra_wasteful',  'not_ultra_wasteful')
clinical_df['expected']       = np.where((clinical_df['match_xGD'].abs() <= 0.1) & (clinical_df['xG'] > 0.5), 'expected',   'not_expected')

# Result vs Expectations
clinical_df['heist']          = np.where((clinical_df['result'] == 'w') & (clinical_df['xG'] < clinical_df['xGA']),            'heist',          'not_heist')
clinical_df['grand_heist']    = np.where((clinical_df['result'] == 'w') & (clinical_df['xGA'] - clinical_df['xG'] >= 1.0),     'grand_heist',    'not_grand_heist')
clinical_df['robbery']        = np.where((clinical_df['result'] == 'l') & (clinical_df['xG'] > clinical_df['xGA']),             'robbery',        'not_robbery')
clinical_df['grand_robbery']  = np.where((clinical_df['result'] == 'l') & (clinical_df['xG'] - clinical_df['xGA'] >= 1.0),     'grand_robbery',  'not_grand_robbery')
clinical_df['dropped_points'] = np.where((clinical_df['result'] == 'd') & (clinical_df['xG'] - clinical_df['xGA'] >= 1.0),     'dropped_points', 'not_dropped_points')
clinical_df['stolen_point']   = np.where((clinical_df['result'] == 'd') & (clinical_df['xGA'] - clinical_df['xG'] >= 1.0),     'stolen_point',   'not_stolen_point')
clinical_df['dominant_win']   = np.where((clinical_df['result'] == 'w') & (clinical_df['xG'] - clinical_df['xGA'] >= 1.5),     'dominant_win',   'not_dominant_win')

# Match Character — deduplicated to avoid double counting
match_level_df = clinical_df[clinical_df['h_a'] == 'h'].copy()
match_level_df['match_total_goals'] = match_level_df['scored'] + match_level_df['missed']
match_level_df['chaos']        = np.where(match_level_df['match_total_goals'] > (match_level_df['xG'] + match_level_df['xGA']) * 1.5, 'chaos',        'not_chaos')
match_level_df['ghost']        = np.where(match_level_df['match_total_goals'] < (match_level_df['xG'] + match_level_df['xGA']) * 0.5, 'ghost',        'not_ghost')
match_level_df['keepers_day']  = np.where((match_level_df['xG'] + match_level_df['xGA'] >= 3.0) & (match_level_df['match_total_goals'] <= 1), 'keepers_day',  'not_keepers_day')
match_level_df['paradise_day'] = np.where((match_level_df['xG'] + match_level_df['xGA'] <= 1.5) & (match_level_df['match_total_goals'] >= 3), 'paradise_day', 'not_paradise_day')

clinical_df = clinical_df.merge(
    match_level_df[['match_id','match_total_goals','chaos','ghost','keepers_day','paradise_day']],
    on='match_id', how='left'
)

# Value columns paired with flag columns
clinical_df['xGD_magnitude']          = clinical_df['match_xGD'].abs().round(3)
clinical_df['dominance_gap']          = (clinical_df['xGA'] - clinical_df['xG']).round(3)
clinical_df['goals_vs_xG_ratio']      = (clinical_df['match_total_goals'] / (clinical_df['xG'] + clinical_df['xGA']).clip(lower=0.01)).round(3)
clinical_df['goals_vs_xG_ratio_team'] = (clinical_df['scored'] / clinical_df['xG'].clip(lower=0.01)).round(3)
clinical_df['gk_save_ratio']          = (1 - (clinical_df['missed'] / clinical_df['xGA'].clip(lower=0.01))).round(3)
clinical_df['xG_total']               = (clinical_df['xG'] + clinical_df['xGA']).round(3)

# Defensive Character
clinical_df['heroic_defence']      = np.where((clinical_df['missed'] == 0) & (clinical_df['xGA'] >= 1.5),  'heroic_defence',      'not_heroic_defence')
clinical_df['routine_clean_sheet'] = np.where((clinical_df['missed'] == 0) & (clinical_df['xGA'] < 0.5),   'routine_clean_sheet', 'not_routine_clean_sheet')
clinical_df['defensive_collapse']  = np.where((clinical_df['missed'] >= 3) & (clinical_df['xGA'] < 1.5),   'defensive_collapse',  'not_defensive_collapse')
clinical_df['gk_worldie']          = np.where((clinical_df['xGA'] - clinical_df['missed']) >= 1.5,          'gk_worldie',          'not_gk_worldie')
clinical_df['gk_nightmare']        = np.where((clinical_df['missed'] - clinical_df['xGA']) >= 1.5,          'gk_nightmare',        'not_gk_nightmare')

# Momentum Based Character
clinical_df['away_win']     = np.where((clinical_df['result'] == 'w') & (clinical_df['h_a'] == 'a'),                                                'away_win',     'not_away_win')
clinical_df['away_heist']   = np.where((clinical_df['result'] == 'w') & (clinical_df['h_a'] == 'a') & (clinical_df['xG'] < clinical_df['xGA']),     'away_heist',   'not_away_heist')
clinical_df['home_bottled'] = np.where((clinical_df['result'] != 'w') & (clinical_df['h_a'] == 'h') & (clinical_df['xG'] > clinical_df['xGA']),     'home_bottled', 'not_home_bottled')

# Full Story
clinical_df['perfect_heist'] = np.where((clinical_df['heist'] == 'heist') & (clinical_df['h_a'] == 'a') & (clinical_df['xGA'] - clinical_df['xG'] >= 1.0), 'perfect_heist', 'not_perfect_heist')
clinical_df['cruel']         = np.where((clinical_df['grand_robbery'] == 'grand_robbery') & (clinical_df['h_a'] == 'a'),                                    'cruel',         'not_cruel')
clinical_df['fortress']      = np.where((clinical_df['heist'] == 'heist') & (clinical_df['h_a'] == 'h'),                                                    'fortress',      'not_fortress')

# Game Tag — always last
clinical_df['game_tag'] = 'Normal'
for col, val, tag in reversed([
    ('perfect_heist', 'perfect_heist', 'Perfect Heist'),
    ('grand_heist',   'grand_heist',   'Grand Heist'),
    ('grand_robbery', 'grand_robbery', 'Grand Robbery'),
    ('cruel',         'cruel',         'Cruel'),
    ('heist',         'heist',         'Heist'),
    ('robbery',       'robbery',       'Robbery'),
    ('ultra_clinical','ultra_clinical','Ultra Clinical'),
    ('dominant_win',  'dominant_win',  'Dominant Win'),
    ('gk_worldie',    'gk_worldie',    'GK Worldie'),
    ('gk_nightmare',  'gk_nightmare',  'GK Nightmare'),
    ('heroic_defence','heroic_defence','Heroic Defence'),
    ('fortress',      'fortress',      'Fortress'),
    ('home_bottled',  'home_bottled',  'Home Bottled'),
    ('clinical',      'clinical',      'Clinical'),
    ('wasteful',      'wasteful',      'Wasteful'),
]):
    clinical_df.loc[clinical_df[col] == val, 'game_tag'] = tag

In [16]:
clinical_df

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,defensive_collapse,gk_worldie,gk_nightmare,away_win,away_heist,home_bottled,perfect_heist,cruel,fortress,game_tag
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Ultra Clinical
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Wasteful
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,defensive_collapse,not_gk_worldie,gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Dominant Win
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Dominant Win
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Dominant Win
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159307,a,1.12443,1.554050,1.12443,1.554050,"{'att': 85, 'def': 13}","{'att': 346, 'def': 24}",7,1,1,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Wasteful
159308,h,1.49032,2.277890,1.49032,2.277890,"{'att': 483, 'def': 15}","{'att': 249, 'def': 19}",8,8,0,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Wasteful
159309,h,1.49032,2.277890,1.49032,2.277890,"{'att': 483, 'def': 15}","{'att': 249, 'def': 19}",8,8,0,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Wasteful
159310,h,1.49032,2.277890,1.49032,2.277890,"{'att': 483, 'def': 15}","{'att': 249, 'def': 19}",8,8,0,...,not_defensive_collapse,not_gk_worldie,not_gk_nightmare,not_away_win,not_away_heist,not_home_bottled,not_perfect_heist,not_cruel,not_fortress,Wasteful


In [17]:
clinical_df.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['h_a', 'xG', 'xGA', 'npxG', 'npxGA', 'ppda', 'ppda_allowed', 'deep',
       'deep_allowed', 'scored', 'missed', 'xpts', 'result', 'date', 'wins',
       'draws', 'loses', 'pts', 'npxGD', 'team_id', 'team_name', 'league',
       'year', 'datetime', 'match_id', 'home_team', 'away_team', 'match_xGD',
       'match_npxGD', 'xGA_diff', 'dominance', 'clinical', 'ultra_clinical',
       'wasteful', 'ultra_wasteful', 'expected', 'heist', 'grand_heist',
       'robbery', 'grand_robbery', 'dropped_points', 'stolen_point',
       'dominant_win', 'match_total_goals', 'chaos', 'ghost', 'keepers_day',
       'paradise_day', 'heroic_defence', 'routine_clean_sheet',
       'defensive_collapse', 'gk_worldie', 'gk_nightmare', 'away_win',
       'away_heist', 'home_bottled', 'perfect_heist', 'cruel', 'fortress',
       'game_tag'],
      dtype='str')>